In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



I0000 00:00:1786365062.989642  476496 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786365063.196443  476496 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786365068.275319  476496 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df = pd.read_csv("../Dataset/df_for_EDA.csv")

In [3]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24"
]

target = "PJME_MW"

X = df[features]
y = df[target]

In [4]:
train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]
X_val = X.iloc[train_size:train_size + val_size]
X_test = X.iloc[train_size + val_size:]

y_train = y.iloc[:train_size]
y_val = y.iloc[train_size:train_size + val_size]
y_test = y.iloc[train_size + val_size:]

In [5]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)

X_val_scaled = X_scaler.transform(X_val)

X_test_scaled = X_scaler.transform(X_test)



In [6]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = y_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
)

In [7]:
sequence_length = 168
forecast_horizon = 24

def create_sequences(X, y, sequence_length, forecast_horizon):

    X_sequences = []
    y_sequences = []

    for i in range(
        sequence_length,
        len(X) - forecast_horizon + 1
    ):

        X_sequences.append(
            X[i-sequence_length:i]
        )

        y_sequences.append(
            y[i:i+forecast_horizon]
        )

    return np.array(X_sequences), np.array(y_sequences)

In [8]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [9]:
print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101447, 168, 13)
y_train: (101447, 24, 1)
X_val: (21588, 168, 13)
y_val: (21588, 24, 1)
X_test: (21590, 168, 13)
y_test: (21590, 24, 1)


# Best Model 

In [10]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [11]:
model = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=13
)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

E0000 00:00:1786365448.907796  476496 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1786365461.471450  476496 cpu_allocator_impl.cc:82] Allocation of 886240992 exceeds 10% of free system memory.


Epoch 1/10
 152/1586 ━━━━━━━━━━━━━━━━━━━━ 5:41 238ms/step - loss: 0.3282 - mae: 0.4180

KeyboardInterrupt: 